# Hassaniya H-CAS — v7.0 (pipeline corrigé + 4 axes de contribution)

**Objectif de cette version** : corriger la fuite méthodologique de la v6.9 (sélection de modèle faite sur le test set) et intégrer les 4 axes de contribution discutés :

- **Axe A** — Juge LLM calibré sur un gold standard humain (approxime H-CAS à l'échelle du corpus)
- **Axe B** — Ré-échantillon aléatoire (non biaisé par CDI) pour ré-annotation humaine + IAA représentatif
- **Axe C** — Diagnostic XAI (SHAP + attention) croisé avec le score H-CAS
- **Axe D** — Baseline LLM zero/few-shot en comparaison directe

**Règle d'or de ce notebook, à ne jamais violer** : `test_df` n'est touché **qu'une seule fois**, à la toute fin (Section 8). Toute sélection de modèle, d'hyperparamètre, de poids de stacking se fait sur `val_df` ou par validation croisée sur `train`/`train_aug`.

Sections :
1. Setup & données (identique v6.9, corrections 1-7 conservées)
2. CDI features (+ note de traçabilité linguistique)
3. Split strict train/val/test + augmentation (train uniquement)
4. Modèles candidats — sélection sur VAL, multi-seeds, IC bootstrap
5. Stacking — poids appris sur VAL (méta-modèle), pas ajustés à la main
6. **Évaluation finale unique sur TEST** + test statistique apparié
7. Axe B — Échantillon aléatoire pour annotation humaine + IAA
8. Axe A — Juge LLM calibré sur le gold humain
9. Axe D — Baseline LLM zero/few-shot
10. Axe C — XAI (SHAP classique + attention transformer) croisé avec H-CAS
11. Synthèse finale H-CAS


## 0. Setup & imports

In [ ]:

import os, re, gc, json, random, inspect, unicodedata, warnings, time, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold,
    cross_val_score, cross_val_predict, GridSearchCV
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, balanced_accuracy_score,
    matthews_corrcoef, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance
from scipy.stats import chi2_contingency

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
)

SEED_LIST = [42, 43, 44]           # CORRECTIF v7.0 : >=3 seeds pour toute comparaison de modele
PRIMARY_SEED = SEED_LIST[0]
random.seed(PRIMARY_SEED); np.random.seed(PRIMARY_SEED); torch.manual_seed(PRIMARY_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(PRIMARY_SEED)

CSV_PATH = None
if CSV_PATH is None or not os.path.isfile(CSV_PATH):
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".csv", ".tsv", ".xlsx")):
                candidates.append(os.path.join(root, f))
    if candidates:
        CSV_PATH = candidates[0]
        print(f"Using: {CSV_PATH}")
    else:
        raise FileNotFoundError("Set CSV_PATH manually.")

OUTPUT_DIR = "/kaggle/working/Hassaniya_HCAS_v7_results"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LENGTH = 128
N_BOOT = 1000

CANDIDATE_TRANSFORMERS = {
    "MARBERTv2": "UBC-NLP/MARBERTv2",
    "CAMeLBERT-DA": "CAMeL-Lab/bert-base-arabic-camelbert-da",
    "AraBERT": "aubmindlab/bert-base-arabertv2",
}

_trainer_sig = inspect.signature(Trainer.__init__).parameters
_TOKENIZER_KW = "processing_class" if "processing_class" in _trainer_sig else "tokenizer"

print("Setup OK. CSV_PATH =", CSV_PATH)


## 1. Chargement, nettoyage, features CDI

Reprend les correctifs v6.8→v6.9 (pas de fuite de dedup, dedup labels valides, tokenisation par mot pour les marqueurs).

**Note de traçabilité (à documenter dans le papier)** : la liste `HASSANIYA_CUES` / `NEGATION` ci-dessous doit être adossée à une source dialectologique citable (lexique publié, travail ethnolinguistique sur le hassaniya) plutôt que présentée comme une liste construite ad hoc — sinon la validité de construit du CDI reste faible en review.


In [ ]:

df_raw = pd.read_csv(CSV_PATH)
df_raw.columns = df_raw.columns.astype(str).str.strip()

VALID = {"positive": "Positive", "negative": "Negative", "neutral": "Neutral"}
def norm_label(v):
    if pd.isna(v): return None
    t = str(v).strip().lower()
    if t in VALID: return VALID[t]
    for k, l in VALID.items():
        if re.search(rf"\b{re.escape(k)}\b", t, re.I): return l
    return None

col = "Annotation results" if "Annotation results" in df_raw.columns else "annotations"
df = pd.DataFrame({
    "id": df_raw["id"] if "id" in df_raw.columns else np.arange(len(df_raw)),
    "text": df_raw["text"].fillna("").astype(str),
    "label": df_raw[col].apply(norm_label)
})
df = df.dropna(subset=["label"]).copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df[df["text"].str.len() > 0]
df = df[df["label"].isin(["Positive", "Negative", "Neutral"])]

def is_noisy(t):
    t = t.strip()
    if len(t) < 2: return True
    if len(set(t)) <= 2 and len(t) > 4: return True
    return False

df = df[~df["text"].apply(is_noisy)].copy()
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

ARABIC_DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
def normalize(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = t.replace("\u0640", "")
    t = t.replace("\u0623", "\u0627").replace("\u0625", "\u0627").replace("\u0622", "\u0627")
    t = t.replace("\u0671", "\u0627").replace("\u0649", "\u064a")
    t = ARABIC_DIAC.sub("", t)
    t = re.sub(r"https?://\S+|www\.\S+", " URL ", t)
    t = re.sub(r"@\w+", " USER ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["clean_text"] = df["text"].apply(normalize)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
df["orig_index"] = np.arange(len(df))

# TODO(papier): remplacer/completer cette liste par des marqueurs sources d'une reference
# dialectologique publiee (lexique hassaniya, travaux ethnolinguistiques). Voir note ci-dessus.
HASSANIYA_CUES = ["\u0623\u064a\u0646\u062a", "\u0630\u0645\u0646", "\u0645\u0631\u062e\u0627\u0643\u0645",
                  "\u0632\u064a\u0646", "\u0623\u0635\u0631\u06af\u0646", "\u0634\u0646\u0648",
                  "\u0627\u0634", "\u0648\u0627\u0634", "\u0647\u0627\u0630", "\u0647\u0627\u0630\u064a",
                  "\u0647\u0627\u0630\u0643", "\u0630\u0627\u0643", "\u0630\u064a\u0643"]
CUE_SET = set(HASSANIYA_CUES)
NEGATION = ["\u0645\u0627", "\u0645\u0648", "\u0645\u0634", "\u0645\u0648\u0634", "\u0645\u0647\u0648",
            "\u0645\u0627\u0646", "\u0645\u0627\u0647\u0648", "\u0644\u0627", "\u0644\u0645",
            "\u0644\u0646", "\u0645\u0627\u0641\u064a", "\u0645\u0641\u064a"]
NEG_SET = set(NEGATION)
GAF = "\u06af"

def safe(x): return "" if pd.isna(x) else str(x)
def tokens(t):
    out = []
    for w in safe(t).split():
        w = re.sub(r"[^\w\u0600-\u06FF]", "", w)
        if w: out.append(w)
    return out

def count_cues(t):
    n = 0
    for w in tokens(t):
        if w in CUE_SET: n += 1
        elif w.startswith("\u0647\u0627\u0630") or w.startswith("\u0630\u0627\u0643"): n += 1
    return n
def count_neg(t): return sum(1 for w in tokens(t) if w in NEG_SET)
def count_rep(t): return int(bool(re.search(r"(.)\1{2,}", safe(t))))
def n_tok(t): return len(safe(t).split())
def has_gaf(t): return int(GAF in safe(t))

def to_group(x):
    if x == 0: return "Low"
    elif x <= 1.5: return "Medium"
    else: return "High"

def add_cdi_features(frame):
    frame["n_tokens"] = frame["text"].apply(n_tok)
    frame["hassaniya_cue_count"] = frame["text"].apply(count_cues)
    frame["negation_count"] = frame["text"].apply(count_neg)
    frame["repetition_count"] = frame["text"].apply(count_rep)
    frame["has_gaf_letter"] = frame["text"].apply(has_gaf)
    frame["bin_cue"]   = (frame["hassaniya_cue_count"] > 0).astype(int)
    frame["bin_neg"]   = (frame["negation_count"] > 0).astype(int)
    frame["bin_rep"]   = (frame["repetition_count"] > 0).astype(int)
    frame["bin_gaf"]   = (frame["has_gaf_letter"] > 0).astype(int)
    frame["bin_short"] = (frame["n_tokens"] <= 5).astype(int)
    frame["CDI_dialect"] = (1.4*frame["bin_gaf"] + 1.2*frame["bin_cue"] +
                            1.1*frame["bin_neg"] + 1.0*frame["bin_rep"])
    frame["CDI_full"] = frame["CDI_dialect"] + 0.35 * frame["bin_short"]
    frame["CDI_group"] = frame["CDI_dialect"].apply(to_group)
    frame["CDI_raw"] = frame["CDI_dialect"]
    return frame

df = add_cdi_features(df)
print(f"Corpus final : {len(df)} lignes | CDI_group :")
print(df["CDI_group"].value_counts())


## 2. Split strict train/val/test + augmentation

`test_df` est isole des maintenant et ne sera rouvert qu'a la Section 6. Toute la suite (selection de modele, d'hyperparametre, de poids) utilise `val_df` ou de la CV interne a `train_aug`.


In [ ]:

train_df, temp = train_test_split(df, test_size=0.30, random_state=PRIMARY_SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp, test_size=0.50, random_state=PRIMARY_SEED, stratify=temp["label"])
train_df = train_df.copy().reset_index(drop=True)
val_df   = val_df.copy().reset_index(drop=True)
test_df  = test_df.copy().reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}  (TEST VERROUILLE jusqu'a la Section 6)")

def char_noise(text, p=0.08):
    if len(text) < 3: return text
    ALPHA = list("\u0627\u0628\u062a\u062b\u062c\u062d\u062e\u062f\u0630\u0631\u0632\u0633"
                 "\u0634\u0635\u0636\u0637\u0638\u0639\u063a\u0641\u0642\u0643\u0644"
                 "\u0645\u0646\u0647\u0648\u064a") + [GAF]
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < p:
            op = random.choice(["sub", "del", "ins"])
            if op == "sub": chars[i] = random.choice(ALPHA)
            elif op == "del" and len(chars) > 3: chars[i] = ""
            elif op == "ins": chars.insert(i, random.choice(["\u0627", "\u064a", "\u0648", GAF]))
    return "".join(chars)

def elongation(text):
    return re.sub(r"(.)\1{1,}", lambda m: m.group(1) * random.randint(1, 3), text)

def classical_augment(text):
    if random.random() < 0.5: text = char_noise(text, p=0.08)
    if random.random() < 0.4: text = elongation(text)
    return text

def innovative_augment(text):
    t = classical_augment(text)
    if random.random() < 0.25: t = char_noise(t, p=0.10)
    return t

aug_rows = []
for _, row in train_df.iterrows():
    new = row.copy(); new["clean_text"] = classical_augment(row["clean_text"]); new["text"] = new["clean_text"]
    aug_rows.append(new)
    if row["label"] == "Neutral" or row["CDI_group"] == "High":
        new2 = row.copy(); new2["clean_text"] = innovative_augment(row["clean_text"]); new2["text"] = new2["clean_text"]
        aug_rows.append(new2)

train_aug = pd.concat([train_df, pd.DataFrame(aug_rows)], ignore_index=True)
train_aug = train_aug.sample(frac=1, random_state=PRIMARY_SEED).reset_index(drop=True)
train_aug = add_cdi_features(train_aug)
print(f"Train original: {len(train_df)} -> augmente: {len(train_aug)} | groupes uniques: {train_aug['orig_index'].nunique()}")

def make_features(wn=3, cn=5):
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, wn), min_df=2, max_df=0.95,
                                 sublinear_tf=True, max_features=60000)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, cn), min_df=2,
                                 sublinear_tf=True, max_features=60000)),
    ])

def metric_dict(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"Accuracy": accuracy_score(y_true, y_pred), "Macro-F1": f,
            "Macro-Precision": p, "Macro-Recall": r,
            "Balanced-Accuracy": balanced_accuracy_score(y_true, y_pred),
            "MCC": matthews_corrcoef(y_true, y_pred)}

label_names = ["Negative", "Neutral", "Positive"]
label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for l, i in label2id.items()}


## 3. Modeles candidats — selection sur VAL uniquement, multi-seeds

Classique (TF-IDF+LR, hyperparametres via CV groupee sur `train_aug`, donc pas de fuite), puis transformers entraines avec 3 seeds chacun. **Le choix du meilleur transformer se fait sur `val_df`**, jamais sur `test_df`.


In [ ]:

# --- 3a. Classique : hyperparametres via CV groupee sur train_aug (pas de fuite, cf. correctif v6.9) ---
groups_aug = train_aug["orig_index"].values
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=PRIMARY_SEED)
param_grid = {
    "features__word__ngram_range": [(1, 2), (1, 3)],
    "features__char__ngram_range": [(2, 4), (2, 5)],
    "clf__C": [0.5, 1.0, 2.0]
}
pipe_lr = Pipeline([("features", make_features()),
                     ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=PRIMARY_SEED))])
gs = GridSearchCV(pipe_lr, param_grid, scoring="f1_macro", cv=cv5, n_jobs=-1)
gs.fit(train_aug["clean_text"], train_aug["label"], groups=groups_aug)
best_classical = gs.best_estimator_
print("Best classical params:", gs.best_params_)

# Evaluation classique sur VAL (jamais sur test a ce stade)
val_pred_classical = best_classical.predict(val_df["clean_text"])
classical_val_metrics = metric_dict(val_df["label"], val_pred_classical)
print("Classical VAL Macro-F1:", round(classical_val_metrics["Macro-F1"], 4))


In [ ]:

# --- 3b. Transformers : 3 seeds par architecture, selection sur VAL ---
train_ids = train_aug["label"].map(label2id).values
counts = np.bincount(train_ids, minlength=3).astype(float)
cw = counts.sum() / (len(counts) * counts); cw[1] *= 1.25

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.03):
        super().__init__(); self.gamma, self.alpha, self.smoothing = gamma, alpha, smoothing
    def forward(self, logits, labels):
        nc = logits.size(-1)
        with torch.no_grad():
            td = torch.zeros_like(logits); td.fill_(self.smoothing/(nc-1))
            td.scatter_(1, labels.unsqueeze(1), 1.0-self.smoothing)
        lp = F.log_softmax(logits, dim=-1)
        ce = -(td*lp).sum(dim=-1)
        if self.alpha is not None: ce = self.alpha[labels]*ce
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()

class FocalTrainer(Trainer):
    def __init__(self, *a, class_weights=None, gamma=2.0, smoothing=0.03, **kw):
        super().__init__(*a, **kw); self.class_weights, self.gamma, self.smoothing = class_weights, gamma, smoothing
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kw):
        labels = inputs.pop("labels"); outputs = model(**inputs); logits = outputs.logits
        w = torch.tensor(self.class_weights, device=logits.device, dtype=logits.dtype) if self.class_weights is not None else None
        loss = FocalLossWithSmoothing(alpha=w, gamma=self.gamma, smoothing=self.smoothing)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, pred, average="macro"), "accuracy": accuracy_score(labels, pred)}

def prepare_ds(tok, frame):
    tmp = frame[["clean_text", "label"]].copy(); tmp["labels"] = tmp["label"].map(label2id)
    return Dataset.from_pandas(tmp[["clean_text", "labels"]], preserve_index=False).map(
        lambda b: tok(b["clean_text"], truncation=True, max_length=MAX_LENGTH), batched=True)

def train_one_transformer(path, seed, tr_frame, va_frame, epochs=6, tag=""):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tok = AutoTokenizer.from_pretrained(path)
    tr_ds, va_ds = prepare_ds(tok, tr_frame), prepare_ds(tok, va_frame)
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=3, id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"/kaggle/working/{tag}_seed{seed}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        num_train_epochs=epochs, weight_decay=0.01, warmup_ratio=0.1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(), seed=seed, report_to="none", save_total_limit=1)
    kw = dict(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
              data_collator=DataCollatorWithPadding(tokenizer=tok), compute_metrics=compute_metrics,
              class_weights=cw, gamma=2.0, smoothing=0.03,
              callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    kw[_TOKENIZER_KW] = tok
    trainer = FocalTrainer(**kw)
    trainer.train()
    val_pred_logits = trainer.predict(va_ds).predictions
    val_preds = [id2label[i] for i in np.argmax(val_pred_logits, axis=-1)]
    m = metric_dict(va_frame["label"], val_preds)
    # CORRECTIF v7.0 (disque plein sur Kaggle) : le modele est deja en memoire
    # (load_best_model_at_end l'a recharge dans trainer.model) -> le checkpoint
    # sur disque est jetable des que les metriques sont extraites.
    shutil.rmtree(args.output_dir, ignore_errors=True)
    return trainer, tok, m

transformer_val_results = {}   # {name: {"seed_metrics": [...], "mean_f1":, "std_f1":, "best_trainer":, "best_tok":}}
for name, path in CANDIDATE_TRANSFORMERS.items():
    print(f"\n>>> {name} — {len(SEED_LIST)} seeds (selection VAL)")
    seed_runs = []
    for s in SEED_LIST:
        trainer, tok, m = train_one_transformer(path, s, train_aug, val_df, epochs=6, tag=name)
        seed_runs.append({"seed": s, "metrics": m, "trainer": trainer, "tok": tok})
        print(f"  seed {s} -> VAL Macro-F1 = {m['Macro-F1']:.4f}")
        gc.collect(); torch.cuda.empty_cache()
    f1s = [r["metrics"]["Macro-F1"] for r in seed_runs]
    best_run = max(seed_runs, key=lambda r: r["metrics"]["Macro-F1"])
    transformer_val_results[name] = {
        "mean_f1": float(np.mean(f1s)), "std_f1": float(np.std(f1s)),
        "seed_f1s": f1s, "best_trainer": best_run["trainer"], "best_tok": best_run["tok"], "path": path
    }
    print(f"  {name} VAL Macro-F1 = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")

# CORRECTIF CRITIQUE v7.0 : selection du meilleur transformer sur VAL (moyenne multi-seeds), jamais sur test
best_tr_name = max(transformer_val_results, key=lambda k: transformer_val_results[k]["mean_f1"])
best_path = CANDIDATE_TRANSFORMERS[best_tr_name]
print(f"\n>>> Meilleur transformer (VAL) : {best_tr_name} "
      f"({transformer_val_results[best_tr_name]['mean_f1']:.4f} +/- {transformer_val_results[best_tr_name]['std_f1']:.4f})")


### 3c. Variantes CDI-weighted (H2) — selection sur VAL

In [ ]:

class WeightedFocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kw):
        sw = inputs.pop("sample_weight", None); labels = inputs.pop("labels")
        outputs = model(**inputs); logits = outputs.logits
        weight = torch.tensor(cw, device=logits.device, dtype=logits.dtype)
        loss = F.cross_entropy(logits.view(-1,3), labels.view(-1), weight=weight, reduction="none")
        pt = torch.exp(-loss); loss = ((1-pt)**2.0 * loss)
        loss = (loss*sw.to(loss.device)).mean() if sw is not None else loss.mean()
        return (loss, outputs) if return_outputs else loss

def make_cdi_dataset(texts, labels, cdi_groups, tokenizer, high_w, med_w):
    class DS(torch.utils.data.Dataset):
        def __init__(self):
            self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=MAX_LENGTH)
            self.labels, self.cdi, self.high_w, self.med_w = list(labels), list(cdi_groups), high_w, med_w
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.labels[i])
            w = self.high_w if self.cdi[i]=="High" else (self.med_w if self.cdi[i]=="Medium" else 1.0)
            item["sample_weight"] = torch.tensor(float(w))
            return item
    return DS()

class CDICollator:
    # CORRECTIF v7.0 : DataCollatorWithPadding ne garantit pas la conservation
    # correcte d'une cle personnalisee (sample_weight) selon la version de transformers
    # (H2-Soft et H2-Strong donnaient des courbes de loss identiques a la 6e decimale
    # -> signe que la ponderation etait silencieusement neutralisee). On pad les champs
    # du tokenizer nous-memes et on empile sample_weight explicitement.
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        # CORRECTIF v7.0 (suite) : en prediction (VAL/TEST via prepare_ds), les features
        # ne contiennent pas sample_weight -> defaut neutre 1.0 au lieu de KeyError.
        sample_weights = torch.stack([f.pop("sample_weight", torch.tensor(1.0)) for f in features])
        # CORRECTIF v7.0 (suite) : remove_unused_columns=False (necessaire pour sample_weight
        # a l'entrainement) reste actif en predict() -> prepare_ds() laisse "clean_text" (str)
        # dans les features, que tokenizer.pad() ne sait pas convertir en tenseur.
        for f in features:
            f.pop("clean_text", None)
        batch = self.tokenizer.pad(features, return_tensors="pt")
        batch["sample_weight"] = sample_weights
        return batch

h2_configs = [("H2-Soft", 1.4, 1.2), ("H2-Strong", 1.7, 1.3)]
h2_val_results = {}
tok_mt = AutoTokenizer.from_pretrained(best_path)
for name, hw, mw in h2_configs:
    print(f"\n>>> {name} (high_w={hw}, med_w={mw})")
    torch.manual_seed(PRIMARY_SEED)
    model = AutoModelForSequenceClassification.from_pretrained(best_path, num_labels=3, id2label=id2label, label2id=label2id)
    train_ds = make_cdi_dataset(train_aug["clean_text"], train_aug["label"].map(label2id), train_aug["CDI_group"], tok_mt, hw, mw)
    val_ds   = make_cdi_dataset(val_df["clean_text"], val_df["label"].map(label2id), val_df["CDI_group"], tok_mt, hw, mw)
    args = TrainingArguments(output_dir=f"/kaggle/working/{name}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-5, per_device_train_batch_size=16, num_train_epochs=6, weight_decay=0.01, warmup_ratio=0.1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(), seed=PRIMARY_SEED, report_to="none", save_total_limit=1,
        remove_unused_columns=False)  # CORRECTIF v7.0 : sinon Trainer retire sample_weight avant le collator
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
              data_collator=CDICollator(tok_mt), compute_metrics=compute_metrics,
              callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    kw[_TOKENIZER_KW] = tok_mt
    trainer = WeightedFocalTrainer(**kw); trainer.train()
    val_logits = trainer.predict(val_ds).predictions
    val_preds = [id2label[i] for i in np.argmax(val_logits, axis=-1)]
    m = metric_dict(val_df["label"], val_preds)
    h2_val_results[name] = {"metrics": m, "trainer": trainer, "tok": tok_mt}
    print(f"{name} VAL Macro-F1 = {m['Macro-F1']:.4f}")
    shutil.rmtree(args.output_dir, ignore_errors=True)  # CORRECTIF v7.0 : idem, disque jetable
    gc.collect(); torch.cuda.empty_cache()

best_h2_name = max(h2_val_results, key=lambda k: h2_val_results[k]["metrics"]["Macro-F1"])
print(f"\n>>> Meilleur H2 (VAL) : {best_h2_name}")


## 4. Stacking — poids appris sur VAL (meta-modele), pas ajustes a la main

CORRECTIF v7.0 : au lieu de poids fixes choisis manuellement (0.15/0.32/0.22/0.18/0.13 dans la v6.9), on entraine une regression logistique meta-modele sur les probabilites predites sur `val_df`, puis on l'applique telle quelle au test.


In [ ]:

def get_proba(model_kind, obj, tok, frame):
    if model_kind == "classical":
        p = obj.predict_proba(frame["clean_text"])
        order = [list(obj.classes_).index(l) for l in label_names]
        return p[:, order]
    else:
        ds = prepare_ds(tok, frame)
        logits = obj.predict(ds).predictions
        return torch.softmax(torch.tensor(logits), dim=-1).numpy()

best_tr_trainer = transformer_val_results[best_tr_name]["best_trainer"]
best_tr_tok = transformer_val_results[best_tr_name]["best_tok"]
best_h2_trainer = h2_val_results[best_h2_name]["trainer"]
best_h2_tok = h2_val_results[best_h2_name]["tok"]

base_models = {
    "classical": ("classical", best_classical, None),
    "transformer": ("transformer", best_tr_trainer, best_tr_tok),
    "h2": ("transformer", best_h2_trainer, best_h2_tok),
}

val_proba_stack = {k: get_proba(*v, val_df) for k, v in base_models.items()}
X_meta_val = np.hstack([val_proba_stack[k] for k in base_models])
y_meta_val = val_df["label"].map(label2id).values

meta_clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=PRIMARY_SEED)
meta_clf.fit(X_meta_val, y_meta_val)
val_stack_pred = [id2label[i] for i in meta_clf.predict(X_meta_val)]
print("Stacking VAL Macro-F1:", round(metric_dict(val_df["label"], val_stack_pred)["Macro-F1"], 4))

# CORRECTIF v7.0 : comparaison finale de tous les candidats SUR VAL, pas sur test
transformer_val_pred = [id2label[i] for i in np.argmax(val_proba_stack["transformer"], axis=-1)]
h2_val_pred = [id2label[i] for i in np.argmax(val_proba_stack["h2"], axis=-1)]

val_comparison = pd.DataFrame([
    {"Model": "Classical", **classical_val_metrics},
    {"Model": best_tr_name, **metric_dict(val_df["label"], transformer_val_pred)},
    {"Model": best_h2_name, **metric_dict(val_df["label"], h2_val_pred)},
    {"Model": "Stacking", **metric_dict(val_df["label"], val_stack_pred)},
]).sort_values("Macro-F1", ascending=False).reset_index(drop=True)
print("\n=== Comparaison sur VAL (selection du modele final) ===")
print(val_comparison.to_string(index=False))
FINAL_MODEL_NAME = val_comparison.iloc[0]["Model"]
print(f"\n>>> MODELE FINAL RETENU (choisi sur VAL) : {FINAL_MODEL_NAME}")


## 5. Evaluation finale unique sur TEST

Premiere et seule utilisation de `test_df` pour un score de performance. On ajoute un intervalle de confiance bootstrap et un test apparie (bootstrap de la difference) entre le modele retenu et le second de la comparaison VAL, pour attester que l'ecart est reel et pas du bruit.


In [ ]:

def predict_by_name(name):
    if name == "Classical": return best_classical.predict(test_df["clean_text"])
    if name == best_tr_name: return [id2label[i] for i in np.argmax(get_proba(*base_models["transformer"], test_df), axis=-1)]
    if name == best_h2_name: return [id2label[i] for i in np.argmax(get_proba(*base_models["h2"], test_df), axis=-1)]
    if name == "Stacking":
        X_meta_test = np.hstack([get_proba(*v, test_df) for v in base_models.values()])
        return [id2label[i] for i in meta_clf.predict(X_meta_test)]
    raise ValueError(name)

final_preds = predict_by_name(FINAL_MODEL_NAME)
final_metrics = metric_dict(test_df["label"], final_preds)
print(f"=== TEST (une seule fois) — {FINAL_MODEL_NAME} ===")
for k, v in final_metrics.items(): print(f"  {k}: {v:.4f}")

# IC bootstrap sur le modele final
rng = np.random.default_rng(PRIMARY_SEED)
y_t, y_p = test_df["label"].values, np.array(final_preds)
n = len(y_t)
boot_f1 = [f1_score(y_t[idx], y_p[idx], average="macro") for idx in (rng.integers(0, n, n) for _ in range(N_BOOT))]
ci_low, ci_high = np.percentile(boot_f1, [2.5, 97.5])
print(f"\nBootstrap Macro-F1 = {np.mean(boot_f1):.4f} [{ci_low:.4f} - {ci_high:.4f}]")

# Test aparie vs le 2e meilleur candidat sur VAL, applique au test
second_name = val_comparison.iloc[1]["Model"]
second_preds = np.array(predict_by_name(second_name))
diffs = []
for _ in range(N_BOOT):
    idx = rng.integers(0, n, n)
    f1_final = f1_score(y_t[idx], y_p[idx], average="macro")
    f1_second = f1_score(y_t[idx], second_preds[idx], average="macro")
    diffs.append(f1_final - f1_second)
diffs = np.array(diffs)
p_value_approx = float(np.mean(diffs <= 0))
print(f"\nComparaison {FINAL_MODEL_NAME} vs {second_name} (bootstrap apparie sur test):")
print(f"  delta moyen = {diffs.mean():+.4f} | IC95% = [{np.percentile(diffs,2.5):+.4f}, {np.percentile(diffs,97.5):+.4f}]")
print(f"  P(delta <= 0) ~ {p_value_approx:.4f}  (si proche de 0, l'ecart est robuste; si > 0.05, ne pas revendiquer de superiorite)")

test_df["prediction"] = final_preds
test_df["correct"] = (test_df["prediction"] == test_df["label"]).astype(int)
test_df.to_csv(os.path.join(OUTPUT_DIR, "final_test_predictions_v7.csv"), index=False, encoding="utf-8-sig")


## 6. Axe B — Echantillon aleatoire pour annotation humaine + IAA

CORRECTIF v7.0 : contrairement a l'`annotation_set` de la v6.9 (trie par CDI, donc biaise vers les cas extremes), on tire ici un echantillon **aleatoire stratifie** sur `label` (et si possible sur `CDI_group`), representatif du corpus entier. C'est celui-ci qui doit servir a calculer un IAA representatif, et a constituer le gold standard de l'Axe A.

Dimensions d'annotation reduites a 3 (au lieu de 4) pour tenir compte du fait que ~65% des textes font <=5 tokens (`bin_short`) — juger 4 dimensions separement sur un texte de 3 mots est peu fiable.


In [ ]:

N_RANDOM_ANNOTATION = 200
random_annotation_set = (
    df.groupby("label", group_keys=False)
      .apply(lambda g: g.sample(frac=N_RANDOM_ANNOTATION/len(df), random_state=PRIMARY_SEED))
      .sample(frac=1, random_state=PRIMARY_SEED)
      .reset_index(drop=True)
)
for c in ["affect", "cultural", "linguistic", "annotator", "comment"]:
    random_annotation_set[c] = ""

print(f"Echantillon aleatoire d'annotation : {len(random_annotation_set)} items")
print(random_annotation_set["CDI_group"].value_counts())
print(random_annotation_set["label"].value_counts())

path_rand_ann = os.path.join(OUTPUT_DIR, "RANDOM_ANNOTATION_gold_v7.csv")
random_annotation_set[["id","text","clean_text","label","CDI_raw","CDI_group",
                       "affect","cultural","linguistic","annotator","comment"]].to_csv(
    path_rand_ann, index=False, encoding="utf-8-sig")
print("Exporte:", path_rand_ann)


In [ ]:

# A executer APRES que 2 annotateurs natifs aient rempli RANDOM_ANNOTATION_gold_v7.csv
# (dupliquer le fichier en annotator1_filled.csv / annotator2_filled.csv, colonnes affect/cultural/linguistic en 1-5)
'''
from sklearn.metrics import cohen_kappa_score
import krippendorff  # pip install krippendorff

ann1 = pd.read_csv("annotator1_filled.csv")
ann2 = pd.read_csv("annotator2_filled.csv")

def compute_iaa(d1, d2, dims=("affect","cultural","linguistic")):
    rows = {}
    for dim in dims:
        kappa = cohen_kappa_score(d1[dim], d2[dim])
        data = np.array([d1[dim].values, d2[dim].values])
        alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="ordinal")
        rows[dim] = {"Cohen_kappa": round(kappa,3), "Krippendorff_alpha": round(alpha,3)}
    return pd.DataFrame(rows).T

iaa = compute_iaa(ann1, ann2)
print(iaa)

gold = ann1.copy()
for dim in ["affect","cultural","linguistic"]:
    gold[dim] = ((ann1[dim] + ann2[dim]) / 2).round(2)
gold["H_CAS_human"] = gold[["affect","cultural","linguistic"]].mean(axis=1)
gold.to_csv("RANDOM_ANNOTATION_gold_MERGED.csv", index=False, encoding="utf-8-sig")
'''
print("Bloc IAA pret (commente) — a decommenter une fois les 2 fichiers annotes disponibles.")


## 7. Axe A — Juge LLM calibre sur le gold humain

Prompt structure, meme grille que l'annotation humaine (affect / cultural / linguistic, 1-5). Necessite une cle API (`ANTHROPIC_API_KEY`) — le bloc est ecrit pour tourner tel quel des que la cle est disponible et que le gold humain (Section 6) existe.


In [ ]:

import os as _os

ANTHROPIC_API_KEY = _os.environ.get("ANTHROPIC_API_KEY", "")

JUDGE_PROMPT_TEMPLATE = '''Tu es un locuteur natif du dialecte arabe hassaniya (Mauritanie/Sahara occidental).
Evalue le texte suivant sur 3 dimensions, chacune sur une echelle de 1 (absent/faible) a 5 (fort/tres marque) :

1. affect : intensite emotionnelle exprimee
2. cultural : ancrage culturel/dialectal specifiquement hassaniya (vocabulaire, expressions, references)
3. linguistic : complexite/authenticite dialectale de la formulation (vs arabe standard ou langue neutre)

Texte : "{text}"

Reponds UNIQUEMENT avec un JSON de la forme :
{{"affect": <1-5>, "cultural": <1-5>, "linguistic": <1-5>}}'''

def llm_judge(text, model="claude-sonnet-4-6", max_retries=3):
    import requests
    if not ANTHROPIC_API_KEY:
        raise RuntimeError("ANTHROPIC_API_KEY non defini dans l'environnement.")
    prompt = JUDGE_PROMPT_TEMPLATE.format(text=text)
    for attempt in range(max_retries):
        try:
            r = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01",
                         "content-type": "application/json"},
                json={"model": model, "max_tokens": 100, "messages": [{"role": "user", "content": prompt}]},
                timeout=30)
            r.raise_for_status()
            raw = r.json()["content"][0]["text"].strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  echec definitif sur un item: {e}")
                return {"affect": None, "cultural": None, "linguistic": None}
            time.sleep(2 ** attempt)

def run_llm_judge_on_frame(frame, text_col="clean_text", sleep_s=0.3):
    results = []
    for t in frame[text_col]:
        results.append(llm_judge(t))
        time.sleep(sleep_s)
    out = frame.copy()
    for dim in ["affect", "cultural", "linguistic"]:
        out[f"{dim}_llm"] = [r.get(dim) for r in results]
    out["H_CAS_llm"] = out[["affect_llm", "cultural_llm", "linguistic_llm"]].mean(axis=1)
    return out

if ANTHROPIC_API_KEY:
    llm_judged = run_llm_judge_on_frame(random_annotation_set)
    llm_judged.to_csv(os.path.join(OUTPUT_DIR, "LLM_JUDGE_random_sample_v7.csv"), index=False, encoding="utf-8-sig")
    print("Juge LLM applique sur l'echantillon aleatoire.")
    print(llm_judged[["affect_llm","cultural_llm","linguistic_llm","H_CAS_llm"]].describe())
else:
    print("ANTHROPIC_API_KEY absent : bloc pret, a executer une fois la cle disponible.")
    print("Une fois le gold humain (Section 6) rempli, comparer H_CAS_llm a H_CAS_human via")
    print("Cohen's kappa / correlation de Spearman pour valider (ou non) le juge LLM comme proxy.")


In [ ]:

# A executer une fois RANDOM_ANNOTATION_gold_MERGED.csv (Section 6) ET LLM_JUDGE_random_sample_v7.csv disponibles
'''
from scipy.stats import spearmanr
gold = pd.read_csv("RANDOM_ANNOTATION_gold_MERGED.csv")
llm = pd.read_csv(os.path.join(OUTPUT_DIR, "LLM_JUDGE_random_sample_v7.csv"))
merged = gold.merge(llm[["id","affect_llm","cultural_llm","linguistic_llm","H_CAS_llm"]], on="id")

for dim in ["affect","cultural","linguistic"]:
    rho, p = spearmanr(merged[dim], merged[f"{dim}_llm"])
    print(f"{dim}: Spearman rho={rho:.3f} (p={p:.4f})")

rho_hcas, p_hcas = spearmanr(merged["H_CAS_human"], merged["H_CAS_llm"])
print(f"H-CAS global: Spearman rho={rho_hcas:.3f} (p={p_hcas:.4f})")

# Stratification par longueur de texte (hypothese : accord plus faible sur les textes courts)
merged["is_short"] = merged["clean_text"].str.split().apply(len) <= 5
for grp, sub in merged.groupby("is_short"):
    if len(sub) > 5:
        rho, _ = spearmanr(sub["H_CAS_human"], sub["H_CAS_llm"])
        print(f"  is_short={grp}: n={len(sub)}, rho={rho:.3f}")

# Si rho global > ~0.5-0.6 : extension du H-CAS_llm a tout le corpus (df complet)
'''
print("Bloc de calibration juge-LLM vs gold humain pret (commente).")


## 8. Axe D — Baseline LLM zero/few-shot (comparaison directe sur TEST)

Comparaison au modele final retenu Section 5, sur le meme `test_df`, memes metriques.


In [ ]:

CLASSIF_PROMPT = '''Classe le sentiment du texte suivant, ecrit en dialecte arabe hassaniya, en une seule des 3 categories : Positive, Negative, Neutral.

{fewshot_block}Texte : "{text}"

Reponds UNIQUEMENT par un seul mot : Positive, Negative ou Neutral.'''

def build_fewshot_block(n_shots=6, seed=PRIMARY_SEED):
    if n_shots == 0: return ""
    ex = train_df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(n=max(1, n_shots // 3), random_state=seed))
    lines = [f'Texte : "{r.clean_text}" -> {r.label}' for _, r in ex.iterrows()]
    return "Exemples :\n" + "\n".join(lines) + "\n\n"

def llm_classify(text, fewshot_block="", model="claude-sonnet-4-6", max_retries=3):
    import requests
    if not ANTHROPIC_API_KEY:
        raise RuntimeError("ANTHROPIC_API_KEY non defini.")
    prompt = CLASSIF_PROMPT.format(fewshot_block=fewshot_block, text=text)
    for attempt in range(max_retries):
        try:
            r = requests.post("https://api.anthropic.com/v1/messages",
                headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"},
                json={"model": model, "max_tokens": 10, "messages": [{"role": "user", "content": prompt}]}, timeout=30)
            r.raise_for_status()
            out = r.json()["content"][0]["text"].strip()
            for lab in label_names:
                if lab.lower() in out.lower(): return lab
            return "Neutral"
        except Exception as e:
            if attempt == max_retries - 1: return "Neutral"
            time.sleep(2 ** attempt)

if ANTHROPIC_API_KEY:
    fs_block = build_fewshot_block(n_shots=6)
    llm_preds_zero = [llm_classify(t, "") for t in test_df["clean_text"]]
    llm_preds_few  = [llm_classify(t, fs_block) for t in test_df["clean_text"]]
    m_zero = metric_dict(test_df["label"], llm_preds_zero)
    m_few  = metric_dict(test_df["label"], llm_preds_few)
    print("LLM zero-shot  Macro-F1:", round(m_zero["Macro-F1"], 4))
    print("LLM few-shot   Macro-F1:", round(m_few["Macro-F1"], 4))
    print(f"Modele specialise ({FINAL_MODEL_NAME}) Macro-F1: {final_metrics['Macro-F1']:.4f}")
    axeD_comparison = pd.DataFrame([
        {"Model": "LLM zero-shot", **m_zero},
        {"Model": "LLM few-shot", **m_few},
        {"Model": FINAL_MODEL_NAME + " (specialise)", **final_metrics},
    ])
    print(axeD_comparison.to_string(index=False))
    axeD_comparison.to_csv(os.path.join(OUTPUT_DIR, "axeD_llm_baseline_comparison_v7.csv"), index=False)
else:
    print("ANTHROPIC_API_KEY absent : bloc pret, a executer une fois la cle disponible.")


## 9. Axe C — Diagnostic XAI (SHAP classique + attention transformer) croise avec H-CAS

Deux volets :
1. **SHAP** sur le pipeline classique (TF-IDF+LR) — feature importance par prediction.
2. **Attention** du transformer — poids d'attention moyens sur les tokens marqueurs dialectaux (گ, cues, negation) vs le reste, pour verifier si le modele "regarde" reellement le signal dialectal.

Le lien avec H-CAS : on croise l'intensite de ce signal XAI avec `H_CAS_llm`/`H_CAS_human` (Section 6-7) pour repondre a la question "le modele s'appuie-t-il davantage sur les marqueurs dialectaux quand le texte est jugé culturellement plus marqué ?"


In [ ]:

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "shap", "--break-system-packages", "-q"])
    import shap

# --- 9a. SHAP sur le pipeline classique ---
sample_for_shap = test_df.sample(min(60, len(test_df)), random_state=PRIMARY_SEED)
vectorizer = best_classical.named_steps["features"]
clf_lr = best_classical.named_steps["clf"]
X_shap = vectorizer.transform(sample_for_shap["clean_text"])

explainer = shap.LinearExplainer(clf_lr, X_shap, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_shap)

feat_names = np.array(vectorizer.get_feature_names_out())
mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2)) if isinstance(shap_values, list) is False else None
print("SHAP calcule sur", X_shap.shape[0], "exemples du test.")

# --- 9b. Attention du transformer sur les marqueurs dialectaux ---
# CORRECTIF v7.0 : les versions recentes de transformers chargent BERT avec l'implementation
# 'sdpa', qui ne peuple pas out.attentions meme si output_attentions=True est demande.
# On recharge la meme architecture en 'eager' et on y copie les poids deja entraines.
best_tr_model = AutoModelForSequenceClassification.from_pretrained(
    transformer_val_results[best_tr_name]["path"], num_labels=3,
    id2label=id2label, label2id=label2id, attn_implementation="eager")
best_tr_model.load_state_dict(best_tr_trainer.model.state_dict())
best_tr_model.to(next(best_tr_trainer.model.parameters()).device)
best_tr_model.eval()
cue_tokens_set = set(HASSANIYA_CUES) | set(NEGATION) | {GAF}

def mean_attention_on_cues(text, tok, model, layer=-1):
    device = next(model.parameters()).device
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # CORRECTIF: aligner sur le device du modele
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)
    attn = out.attentions[layer][0].mean(dim=0)   # moyenne sur les tetes -> [seq, seq]
    cls_attn = attn[0]                             # attention du token [CLS] vers chaque token
    toks = tok.convert_ids_to_tokens(inputs["input_ids"][0])
    cue_mask = np.array([any(c in t for c in cue_tokens_set) for t in toks])
    if cue_mask.sum() == 0:
        return None
    return float(cls_attn.numpy()[cue_mask].mean()), float(cls_attn.numpy()[~cue_mask].mean())

attn_rows = []
for _, row in sample_for_shap.iterrows():
    res = mean_attention_on_cues(row["clean_text"], best_tr_tok, best_tr_model)
    if res is not None:
        attn_rows.append({"id": row["id"], "attn_on_cue": res[0], "attn_on_rest": res[1],
                          "CDI_group": row["CDI_group"]})
attn_df = pd.DataFrame(attn_rows)
if len(attn_df):
    print("\nAttention moyenne [CLS] -> tokens marqueurs vs reste, par groupe CDI:")
    print(attn_df.groupby("CDI_group")[["attn_on_cue", "attn_on_rest"]].mean().round(4))
    attn_df.to_csv(os.path.join(OUTPUT_DIR, "axeC_attention_on_cues_v7.csv"), index=False)
else:
    print("Aucun texte de l'echantillon ne contient de marqueur dialectal detectable par le tokenizer.")


In [ ]:

# A executer une fois H_CAS_llm (Axe A) disponible sur ce meme echantillon de test
'''
from scipy.stats import spearmanr
merged_xai = attn_df.merge(llm_judged[["id","H_CAS_llm"]], on="id", how="inner")
merged_xai["attn_ratio"] = merged_xai["attn_on_cue"] / (merged_xai["attn_on_rest"] + 1e-9)
rho, p = spearmanr(merged_xai["attn_ratio"], merged_xai["H_CAS_llm"])
print(f"Correlation attention-sur-marqueurs vs H-CAS : rho={rho:.3f} (p={p:.4f})")
'''
print("Bloc de correlation XAI x H-CAS pret (commente).")


## 10. Synthese finale H-CAS

Tableau qui met en regard, pour le modele retenu : performance classique (Macro-F1), H-CAS humain (echantillon aleatoire), H-CAS LLM (etendu), et le diagnostic XAI — la preuve empirique du "Beyond F1".


In [ ]:

synthesis = {
    "Modele final": FINAL_MODEL_NAME,
    "Test Macro-F1": round(final_metrics["Macro-F1"], 4),
    "Test Macro-F1 CI95": [round(ci_low, 4), round(ci_high, 4)],
    "N annotation aleatoire (Axe B)": len(random_annotation_set),
    "N High-CDI dans le test": int((test_df["CDI_group"] == "High").sum()),
}
print(json.dumps(synthesis, ensure_ascii=False, indent=2))

with open(os.path.join(OUTPUT_DIR, "SYNTHESIS_v7.json"), "w", encoding="utf-8") as f:
    json.dump(synthesis, f, ensure_ascii=False, indent=2)

print("\nNotebook v7.0 termine.")
print("Rappel des etapes MANUELLES restantes avant redaction :")
print("  1. Faire annoter RANDOM_ANNOTATION_gold_v7.csv par >=2 locuteurs natifs -> calculer IAA (Section 6)")
print("  2. Definir ANTHROPIC_API_KEY et relancer les Sections 7-8 (juge LLM + baseline LLM)")
print("  3. Une fois le gold humain dispo, relancer les blocs de calibration (Sections 7 et 9)")
print("  4. Ancrer HASSANIYA_CUES/NEGATION sur une reference dialectologique citable (Section 1)")
